# Main Training Script Overview

This file serves as the **main execution file** for running experiments with the XGBoost model. It automates model training and evaluation across 10 datasets and 5 train–test splits.

## Dependencies
- Core helper functions are defined in **`functions.py`**.  
- Input datasets must be generated beforehand using the **data preprocessing step**.

## What this file does
- **Dataset loading:** Imports the 10 datasets produced during data preprocessing.  
- **Experiment repetitions:** For each dataset, performs 5 fold cross validation (total of 50 runs).  
- **Model training:** Trains an **XGBoost model** on each split.  
- **Outputs:** Saves evaluation and interpretation results, including:
  - **ROC curves (CSV format) (weighted vs unweighted plots - not used in paper)**  
  - **Feature importance scores (CSV format)**  
  - **SHAP bee plots**  

# run below cell to get global variables


In [ ]:
from functions import *

# load the data
df = pd.read_csv(r"../Data Cleaning/results/merged_labeled.csv", index_col="eid")
df["Status"] = df["Status"].map({"healthy": 0, "dementia": 1, "AD": 2})


CAT_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Alcohol Consumption",
    "Smoking Status",
    "Antihypertensive usage",
]

for col in CAT_FEATURES:
    if col in df.columns:
        df[col] = df[col].astype("category")

NUM_FEATURES = df.columns.difference(CAT_FEATURES + ["Status"])

MAIN_FEATURES = [
    "Age",
    "Sex",
    "Educational status",
    "Diabetes",
    "Spherical equivalent",
    "Systolic blood pressure",
    "Diastolic blood pressure",
    "Antihypertensive usage",
    "Alcohol Consumption",
    "Smoking Status",
    "BMI",
    "mRNFL thickness",
    "mGCIPL thickness",
    "Status",
]
df_main = df.loc[:, MAIN_FEATURES]
MAIN_NUM_FEATURES = df_main.columns.difference(CAT_FEATURES + ["Status"])
MAIN_CAT_FEATURES = CAT_FEATURES

In [ ]:
df.info()

In [ ]:
# combined data files

# mm stands for missing matched

ldf_mm_missing = []
ldf_mm_missing_core = []

ldf_mm_knn = []
ldf_mm_knn_core = []

ldf_mm_randomfill = []
ldf_mm_randomfill_core = []

ldf_age_matched_mm_missing = []
ldf_age_matched_mm_missing_core = []

ldf_age_matched_mm_randomfill = []
ldf_age_matched_mm_randomfill_core = []

ldf_age_matched_mm_knn = []
ldf_age_matched_mm_knn_core = []


output_path = r"./results_combined/"

for s in range(1, 11):
    input_path = r"../Data Cleaning/results_{}/".format(s)

    df_mm_randomfill = pd.read_csv(
        input_path + r"missing_matched_randomfill.csv", index_col="eid"
    )
    df_mm_knn = pd.read_csv(input_path + r"missing_matched_knn.csv", index_col="eid")
    df_mm_missing = pd.read_csv(
        input_path + r"missing_matched_missing.csv", index_col="eid"
    )

    df_age_matched_mm_randomfill = pd.read_csv(
        input_path + r"missing_matched_age_matched_randomfill.csv", index_col="eid"
    )
    df_age_matched_mm_knn = pd.read_csv(
        input_path + r"missing_matched_age_matched_knn.csv", index_col="eid"
    )
    df_age_matched_mm_missing = pd.read_csv(
        input_path + r"missing_matched_age_matched_missing.csv", index_col="eid"
    )

    df_mm_knn_core = df_mm_knn.loc[:, MAIN_FEATURES].copy()
    df_mm_randomfill_core = df_mm_randomfill.loc[:, MAIN_FEATURES].copy()
    df_mm_missing_core = df_mm_missing.loc[:, MAIN_FEATURES].copy()

    df_age_matched_mm_knn_core = df_age_matched_mm_knn.loc[:, MAIN_FEATURES].copy()
    df_age_matched_mm_randomfill_core = df_age_matched_mm_randomfill.loc[
        :, MAIN_FEATURES
    ].copy()
    df_age_matched_mm_missing_core = df_age_matched_mm_missing.loc[
        :, MAIN_FEATURES
    ].copy()

    # append to the lists

    ldf_mm_knn.append(df_mm_knn)
    ldf_mm_knn_core.append(df_mm_knn_core)

    ldf_mm_randomfill.append(df_mm_randomfill)
    ldf_mm_randomfill_core.append(df_mm_randomfill_core)

    ldf_mm_missing.append(df_mm_missing)
    ldf_mm_missing_core.append(df_mm_missing_core)

    ldf_age_matched_mm_knn.append(df_age_matched_mm_knn)
    ldf_age_matched_mm_knn_core.append(df_age_matched_mm_knn_core)

    ldf_age_matched_mm_randomfill.append(df_age_matched_mm_randomfill)
    ldf_age_matched_mm_randomfill_core.append(df_age_matched_mm_randomfill_core)

    ldf_age_matched_mm_missing.append(df_age_matched_mm_missing)
    ldf_age_matched_mm_missing_core.append(df_age_matched_mm_missing_core)

# missing matched

## missing

In [ ]:
df_combined = pd.concat(ldf_mm_missing, ignore_index=False)
df_combined.drop_duplicates(inplace=True)
df_combined.to_csv(r"./combined_mm_missing.csv")

In [ ]:
ldf_mm_missing[0].isnull().sum()

In [ ]:
ldf_mm_knn[0]["Status"].count()

In [ ]:
filtered_cohort_profile = pd.concat(ldf_mm_knn, ignore_index=False)
filtered_cohort_profile.drop_duplicates(inplace=True)
filtered_cohort_profile["Status"].describe()

In [ ]:
ldf_age_matched_mm_knn[0]["Status"].count()

In [ ]:
age_filtered_cohort_profile = pd.concat(ldf_age_matched_mm_knn, ignore_index=False)
age_filtered_cohort_profile.drop_duplicates(inplace=True)
age_filtered_cohort_profile["Status"].describe()

In [ ]:
from matplotlib.ticker import MultipleLocator

ad_years = pd.read_csv("ad_years.csv")
dd = ad_years["ad_after0"]

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        # Arial first; Arial-metric / sans-serif fallbacks so the figure stays
        # compliant even without mscorefonts (see requirements.txt for install).
        "font.sans-serif": [
            "Arial",
            "Liberation Sans",
            "Helvetica",
            "Nimbus Sans",
            "DejaVu Sans",
        ],
        "font.size": 13,
        "axes.labelweight": "normal",  # normal-weight axis labels (Fig2 labels not bold)
        "axes.linewidth": 0.8,
        "axes.edgecolor": "#444444",
        "svg.fonttype": "none",
        "pdf.fonttype": 42,  # editable text in the PDF
    }
)

BAR_COLOR = "#2C6E8F"
EDGE_COLOR = "#1B4A61"
MEAN_COLOR = "#B5452F"

# ----------------------------------------------------------------------
# BINS  (left-closed: [0,2), [2,4), ...)
# ----------------------------------------------------------------------
min_val = int(np.floor(dd.min()))
max_val = int(np.ceil(dd.max()))
# round the lower edge down to an even number so bins align on even years
min_edge = min_val - (min_val % 2)
bins = np.arange(min_edge, max_val + 2, 2)

counts, edges = np.histogram(dd, bins=bins)
centers = edges[:-1] + np.diff(edges) / 2
mean_val = dd.mean()
std_val = dd.std()

# ----------------------------------------------------------------------
# PLOT
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

bars = ax.bar(
    centers,
    counts,
    width=np.diff(edges) * 0.92,
    color=BAR_COLOR,
    edgecolor=EDGE_COLOR,
    linewidth=0.8,
    zorder=3,
)

# value labels on top of each bar
for c, n in zip(centers, counts):
    if n > 0:
        ax.text(
            c,
            n + max(counts) * 0.015,
            str(int(n)),
            ha="center",
            va="bottom",
            fontsize=11,
            color="#333333",
            zorder=4,
        )

# mean line + label
ax.axvline(mean_val, color=MEAN_COLOR, linestyle=(0, (5, 3)), linewidth=1.6, zorder=5)
ax.text(
    mean_val,
    max(counts) * 1.2,
    f"  mean = {mean_val:.2f} yr\n  (SD = {std_val:.2f})",
    ha="left",
    va="top",
    fontsize=11,
    color=MEAN_COLOR,
    zorder=6,
)

# axis labels (real meaning, not the variable name)
ax.set_xlabel("Years from eye measurement to AD diagnosis", labelpad=8)
ax.set_ylabel("Number of AD cases", labelpad=8)

# x ticks: bin-edge interval labels, centered under each bar, horizontal
edge_labels = [f"[{int(edges[i])}, {int(edges[i + 1])})" for i in range(len(edges) - 1)]
ax.set_xticks(centers)
ax.set_xticklabels(edge_labels, fontsize=11)

# y ticks every 20, light minor ticks every 10
ax.yaxis.set_major_locator(MultipleLocator(20))
ax.yaxis.set_minor_locator(MultipleLocator(10))
ax.set_ylim(0, max(counts) * 1.18)
ax.set_xlim(edges[0] - 1, edges[-1] + 1)

# grid behind bars only on y
ax.yaxis.grid(True, which="major", color="#000000", alpha=0.10, linewidth=0.7, zorder=0)
ax.yaxis.grid(True, which="minor", color="#000000", alpha=0.05, linewidth=0.5, zorder=0)
ax.set_axisbelow(True)

# despine top/right
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", color="#444444", length=4, width=0.8)
ax.tick_params(axis="x", length=0)  # no x tick marks; labels are intervals

fig.tight_layout()
fig.savefig("ad_hist.pdf", bbox_inches="tight")
fig.savefig("ad_hist.png", bbox_inches="tight", dpi=200)
print(f"counts = {counts.tolist()}")
print(f"mean = {mean_val:.3f}, sd = {std_val:.3f}, n = {len(dd)}")

In [ ]:
process_dementia_vs_healthy(
    ldf_mm_missing,
    "missing_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_mm_missing,
    "missing_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_mm_missing_core,
    "missing_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_mm_missing_core,
    "missing_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_age_matched_mm_missing,
    "missing_age_matched_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_age_matched_mm_missing,
    "missing_age_matched_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_age_matched_mm_missing_core,
    "missing_age_matched_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_age_matched_mm_missing_core,
    "missing_age_matched_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

## randomfill

In [ ]:
process_dementia_vs_healthy(
    ldf_mm_randomfill,
    "randomfill_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_mm_randomfill,
    "randomfill_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_mm_randomfill_core,
    "randomfill_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_mm_randomfill_core,
    "randomfill_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_age_matched_mm_randomfill,
    "randomfill_age_matched_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_age_matched_mm_randomfill,
    "randomfill_age_matched_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_age_matched_mm_randomfill_core,
    "randomfill_age_matched_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_age_matched_mm_randomfill_core,
    "randomfill_age_matched_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

## KNN

In [ ]:
process_dementia_vs_healthy(
    ldf_mm_knn,
    "knn_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_mm_knn,
    "knn_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_mm_knn_core,
    "knn_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_mm_knn_core,
    "knn_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_age_matched_mm_knn,
    "knn_age_matched_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_age_matched_mm_knn,
    "knn_age_matched_mm",
    NUM_FEATURES,
    CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=False,
    output_path=output_path,
)

In [ ]:
process_dementia_vs_healthy(
    ldf_age_matched_mm_knn_core,
    "knn_age_matched_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=True,
    output_path=output_path,
)

process_ad_vs_healthy(
    ldf_age_matched_mm_knn_core,
    "knn_age_matched_mm - core",
    MAIN_NUM_FEATURES,
    MAIN_CAT_FEATURES,
    is_final=True,
    is_shapley=True,
    is_csv=False,
    output_path=output_path,
)